In [2]:
import yfinance as yf
import pandas as pd

tickers = {
    "TCS.NS": "TCS",
    "INFY.NS": "Infosys",
    "WIPRO.NS": "Wipro",
    "HCLTECH.NS": "HCL Technologies",
    "TECHM.NS": "Tech Mahindra",
    "LTM.NS": "LTIMindtree",
    "PERSISTENT.NS": "Persistent Systems",
    "COFORGE.NS": "Coforge",
    "MPHASIS.NS": "Mphasis",
    "LTTS.NS": "L&T Technology Services"
}

all_prices = []
for ticker, name in tickers.items():
    data = yf.download(ticker, start="2021-04-01", end="2026-04-01")
    data = data[["Close"]].copy()
    data.columns = ["close_price"]         
    data["company"] = name
    data["date"] = data.index
    all_prices.append(data)

stock_prices = pd.concat(all_prices, ignore_index=True)
stock_prices.to_csv("stock_prices.csv", index=False)
print(stock_prices.head())
print(stock_prices.shape)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


   close_price company       date
0  2730.089355     TCS 2021-04-01
1  2793.834473     TCS 2021-04-05
2  2816.088623     TCS 2021-04-06
3  2821.867920     TCS 2021-04-07
4  2861.504883     TCS 2021-04-08
(12350, 3)


In [5]:
import pandas as pd


financials = pd.read_csv(r"C:\Users\Asus\Downloads\company_financials_data.csv")
stock_prices = pd.read_csv("stock_prices.csv")
stock_prices["date"] = pd.to_datetime(stock_prices["date"])


stock_prices["year"] = stock_prices["date"].apply(
    lambda d: d.year if d.month < 4 else d.year + 1
)


avg_price = (
    stock_prices.groupby(["company", "year"])["close_price"]
    .mean()
    .reset_index()
    .rename(columns={"close_price": "avg_stock_price"})
)


merged = financials.merge(avg_price, on=["company", "year"], how="left")


merged = merged.sort_values(["company", "year"])
merged["profit_margin"] = merged["net_profit"] / merged["revenue"]
merged["yoy_revenue_growth"] = merged.groupby("company")["revenue"].pct_change() * 100

merged.to_csv("company_financials_combined.csv", index=False)
print(merged.head(10))
print(merged.shape)

             company  year   revenue  net_profit  operating_profit  \
35           Coforge  2022    6432.0       661.7               NaN   
36           Coforge  2023    8015.0       812.0               NaN   
37           Coforge  2024    9009.0       808.0               NaN   
38           Coforge  2025   12051.0       812.1               NaN   
39           Coforge  2026   16402.7      1555.7               NaN   
15  HCL Technologies  2022       NaN     13499.0           20529.0   
16  HCL Technologies  2023       NaN     14851.0           22628.0   
17  HCL Technologies  2024       NaN     15710.0           24198.0   
18  HCL Technologies  2025  117055.0     17390.0           25504.0   
19  HCL Technologies  2026  130144.0     16642.0               NaN   

    total_debt    eps  avg_stock_price  profit_margin  yoy_revenue_growth  
35         NaN    NaN       876.397965       0.102876                 NaN  
36         NaN    NaN       734.512088       0.101310           24.611318  
3

C:\Users\Asus\AppData\Local\Temp\ipykernel_10268\4186004459.py:27: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  merged["yoy_revenue_growth"] = merged.groupby("company")["revenue"].pct_change() * 100


In [6]:
print(merged.head(10))

             company  year   revenue  net_profit  operating_profit  \
35           Coforge  2022    6432.0       661.7               NaN   
36           Coforge  2023    8015.0       812.0               NaN   
37           Coforge  2024    9009.0       808.0               NaN   
38           Coforge  2025   12051.0       812.1               NaN   
39           Coforge  2026   16402.7      1555.7               NaN   
15  HCL Technologies  2022       NaN     13499.0           20529.0   
16  HCL Technologies  2023       NaN     14851.0           22628.0   
17  HCL Technologies  2024       NaN     15710.0           24198.0   
18  HCL Technologies  2025  117055.0     17390.0           25504.0   
19  HCL Technologies  2026  130144.0     16642.0               NaN   

    total_debt    eps  avg_stock_price  profit_margin  yoy_revenue_growth  
35         NaN    NaN       876.397965       0.102876                 NaN  
36         NaN    NaN       734.512088       0.101310           24.611318  
3

In [7]:
print(merged.shape)

(50, 10)


In [8]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://localhost\\SQLEXPRESS/it_financials?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

merged.to_sql("company_financials", engine, if_exists="replace", index=False)
print("Loaded successfully")

Loaded successfully
